# 🧱 01 — Ledgers, keys, and shapes

*The three pieces of table-setting every later chapter stands on: a record nobody can
rewrite, identities nobody can fake, and messages nobody can misread.*

Chapter 00 ended with Ada and Bell unable to trade because neither could safely move
first. Before chapter 03 builds the machine that fixes that, we need three humble
ingredients on the table — and this chapter is **deliberately light**: just enough of
each, built by hand, so that nothing later feels like magic. (For the paper, this is
background material — the primer a reader needs, not a contribution.)

**You need:** nothing. The short final section wants [Foundry](https://getfoundry.sh)
installed (`forge build --root contracts` once), and politely skips if it's missing.

**How to work through it:** run every cell, in order. When you hit a **✏️ Your turn**,
write your answer in the scaffold cell *before* opening the fold-out solution. 🧭
Decision boxes and the closing 📝 section work exactly as in the rest of the course:
they collect what this chapter contributes to the paper.

## 0 · The question of this chapter

Three things have to exist before two strangers can even *talk about* trading:

1. **A ledger nobody owns** — a record of who has what, that neither party (nor anyone
   else) can quietly rewrite.
2. **Identities nobody can fake** — a way for "Ada" and "Bell" to be *checkable* names,
   not just strings anyone can type.
3. **Words nobody can misread** — deal terms precise enough that a computer on the other
   side can't misunderstand them.

Each gets a section. Chapter 03 then spends all three without apology.

## 1 · The ledger nobody owns

Strip away every buzzword and a ledger is a table of who has how much. Chapter 00's
version, again — a Python dict and a transfer rule:

In [ ]:
balances = {"Ada": 100, "Bell": 20}

def pay(sender, receiver, amount):
    if balances[sender] < amount:
        raise ValueError(f"{sender} only has {balances[sender]} TOK")
    balances[sender] -= amount
    balances[receiver] += amount

pay("Ada", "Bell", 10)
print(balances)

The problem isn't the arithmetic — it's **custody**. This dict lives in *somebody's*
computer, and whoever that somebody is can edit it without calling `pay` at all:

In [ ]:
# Mallory hosts the ledger. Mallory is having a bad quarter.
balances["Mallory"] = 1_000_000
balances["Ada"] -= 50

print(balances)
print("No rule was broken, because the rules only exist inside pay() —")
print("and nothing forces the host to go through pay().")

Two fixes are needed, and they're separable. First: make *history* tamper-evident, so
edits can at least be **detected**. Second: get the ledger off any single computer, so
detection has teeth. The tool for the first is a **hash** — a fingerprint function: it
eats any data and produces a short string with the property that changing even one
character of the input produces a completely different fingerprint:

In [ ]:
import hashlib

def fingerprint(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]   # short for readability

print(fingerprint("Ada pays Bell 10 TOK"))
print(fingerprint("Ada pays Bell 11 TOK"))   # one character changed
print(fingerprint("Ada pays Bell 10 TOK"))   # same input, same fingerprint, always

A fingerprint of one entry proves *that entry* wasn't edited. The trick that turns
fingerprints into a tamper-evident **history** is chaining: every block of history
includes the fingerprint of the block before it. Now the blocks are welded together —
edit any block and its fingerprint changes, which breaks the *next* block's stored
copy of it, which breaks the next, all the way to the end:

In [ ]:
def make_block(prev_hash, data):
    return {"prev": prev_hash, "data": data,
            "hash": fingerprint(prev_hash + str(data))}

chain = [make_block("genesis", ("Ada", "Bell", 10))]
for deal in [("Carol", "Bell", 5), ("Ada", "Carol", 2), ("Bell", "Ada", 1)]:
    chain.append(make_block(chain[-1]["hash"], deal))

def verify(chain):
    prev = "genesis"
    for i, block in enumerate(chain):
        ok = block["prev"] == prev and block["hash"] == fingerprint(prev + str(block["data"]))
        print(f"block {i}: {block['data']}   {'✓' if ok else '✗ BROKEN'}")
        prev = block["hash"]

verify(chain)

In [ ]:
# Mallory rewrites history: block 1's 5 TOK becomes 500.
chain[1]["data"] = ("Carol", "Bell", 500)

verify(chain)
print("\nOne edit, and every block from the edit onward screams.")

**✏️ Your turn 1 — the patient forger**

Mallory isn't done. Editing the data broke the chain — but the chain is just data too.
Starting from the tampered block 1, **recompute** every `hash` and `prev` from there to
the end, so `verify` passes again. (A loop over `chain[1:]` rebuilding each block.)
Then answer in a comment: **what did we actually gain from chaining, if a forger can
just re-weld it?**

In [ ]:
# ...recompute the chain from block 1 onward so verify(chain) is all ✓ again...

<details><summary>✅ Solution 1 — peek only after trying</summary>

```python
prev = chain[0]["hash"]
for i in range(1, len(chain)):
    chain[i] = make_block(prev, chain[i]["data"])
    prev = chain[i]["hash"]
verify(chain)   # all ✓ — history rewritten, seamlessly
```

What chaining bought us is **not** immutability — it's that a forgery can't be *local*.
To change one old entry Mallory had to rewrite every block after it. On one computer
that's a millisecond of work, so on one computer the chain is merely tidy, not safe.
The gain appears the moment **more than one computer holds a copy**: now Mallory's
rewritten tail disagrees with everyone else's, and the forgery is visible precisely
*because* it couldn't stay local.

</details>

That last sentence is the whole idea of a **blockchain**, minus engineering: thousands
of computers each hold the chain, each re-checks every new block, and a rule (consensus)
picks whose version wins — making "rewrite the tail everywhere at once" the attack you'd
need, which is somewhere between expensive and hopeless. That's all the depth this
course needs: **the ledger is a chained history that many machines cross-check, so
nobody — host, Ada, Bell, Mallory — can quietly edit it.** The programs that run *on*
that ledger (chapter 03's vending machine) inherit exactly this tamper-resistance.

> **🧭 Decision (pragmatic) — one local node, consensus bought off the shelf**
>
> This project runs its ledger as a single local Anvil node (you'll meet it in §4).
> Real deployments spread the ledger over many machines and pay for consensus in
> latency and fees; we don't study any of that — consensus is *assumed to work*,
> exactly as advertised, and the prototype buys it off the shelf. Chosen because the
> mechanism under test is settlement-for-entitlement, not distributed agreement.
> **In the paper:** §8.4 Measurement validity — Anvil mines instantly, so public-chain
> latency is extrapolated, not measured.

## 2 · Identities nobody can fake

Chapter 00 left a thread hanging: a history entry saying `("Bell", "Mallory", 50)` is
just a tuple — nothing about it proves *Bell* produced it. What's missing is something
only Bell can create and everyone can check. Here is the shape of the fix:

- Every party invents a **private key**: a huge random secret number. Never shown to
  anyone.
- From the key, a public **address** is derived: a short string safe to publish — the
  account number people pay.
- With the key, its holder can produce a **signature** over any exact piece of data:
  a stamp anyone can check against the *address* (no secret needed to verify), that
  nobody without the key can forge, and that stops matching if the data changes by
  one bit.

*How* signatures achieve that is chapter 04's whole subject — today we only need the
interface. But keys and addresses we can touch right now:

In [ ]:
from eth_account import Account

acct = Account.create()                      # a brand-new identity: one random number
print("private key →", acct.key.hex())
print("address     →", acct.address)
print("\nRun this cell again — a different key, a different address, every time.")

An "account" on a chain is *nothing but this*: a key pair. No registration, no
database row, no permission. And the cast you've been reading about is exactly four
such key pairs — Anvil's well-known development keys (public constants for labs;
real keys are secrets):

In [ ]:
from a2a_interfaces import fixtures as fx        # the canonical example, one source of truth
from chainmcp.testing import ANVIL_KEYS          # the lab cast's keys (dev keys, not secrets)

ada = Account.from_key(ANVIL_KEYS["ada"])
print("derived from Ada's key  →", ada.address)
print("fixtures.ADA            →", fx.ADA)
print("same?                   →", ada.address == fx.ADA)

bell = Account.from_key(ANVIL_KEYS["bell"])
print("\nderived from Bell's key →", bell.address)
print("fixtures.BELL           →", fx.BELL)

The `0xf39F…` string you've seen since chapter 00 was Ada's *key*, wearing its public
face, all along. One small aside before it puzzles you: the odd MiXeD cAsE in addresses
is deliberate — the capitalization pattern is a built-in typo detector (it's derived
from the address's own hash, so a mistyped address almost surely has the wrong
capitalization and tools reject it):

In [ ]:
print("as published:", fx.ADA)
print("lowercased  :", fx.ADA.lower(), " ← same account, but the typo-check is gone")

> **🧭 Decision (principled) — private keys live in exactly one component per agent**
>
> **Chosen:** each agent has a *key custodian* (the `chainmcp` package — an MCP tool
> server); it alone holds the key and signs. No other package reads, receives, stores,
> or logs a key. The provider's controller — the code that will guard the router in
> chapter 05 — **verifies signatures but never signs.**
> **Alternatives:** (a) the agent process holds its own key (fewer moving parts);
> (b) a shared signing service holds everyone's keys; (c) the controller signs too,
> for convenience.
> **Why:** whoever holds a key can spend the money and issue the promises of its
> owner. An LLM-driven agent holding a key means a prompt-injected agent can be robbed;
> a shared service is one breach away from every identity at once; a signing controller
> collapses the verifier/spender boundary — the bouncer should check tickets, not be
> *able to print them*. One key, one custodian, smallest possible blast radius.
> **In the paper:** §4.6, trust boundaries — "keys live in exactly one component per
> agent; the controller verifies signatures but never signs."

**✏️ Your turn 2 — complete the cast**

Two cast members remain: `carol` (another honest customer) and `mallory` (the
freeloader). Derive both addresses from `ANVIL_KEYS` and print them. Then answer in a
comment: **if Mallory publishes his address, what can other people do with it — and
what can they still not do?**

In [ ]:
# carol_address = ...
# mallory_address = ...

<details><summary>✅ Solution 2 — peek only after trying</summary>

```python
carol_address = Account.from_key(ANVIL_KEYS["carol"]).address
mallory_address = Account.from_key(ANVIL_KEYS["mallory"]).address
print("Carol  :", carol_address)
print("Mallory:", mallory_address)
```

With Mallory's *address*, anyone can send him money, check his balance, and verify
signatures he makes. What they still cannot do is the only thing that matters: **act as
him** — spend from the address or sign in its name. That asymmetry (publish the
address, guard the key) is the entire identity model, and it's why the decision box
above is so strict about where keys live.

</details>

## 3 · Words nobody can misread

Third ingredient. Ada and Bell are programs; their "conversation" is data. And data
between strangers has a failure mode subtler than forgery: **being understood
differently by each side.** Here's a deal as a plain dict, with three time bombs in it —
run the cell and note that Python accepts all of them without a murmur:

In [ ]:
deal = {
    "provider": "Bell",
    "capacity": 50,          # 💣 fifty WHAT? Ada means Mbit/s; Bell's router speaks bit/s
    "price": 10.0,           # 💣 a float — and money as floats goes wrong (below)
    # 💣 "end" is missing — nobody will notice until someone reads it
    "start": 1_757_944_800,
}

print("accepted:", deal)
print("The dict is happy. Every bug above is now the *reader's* problem, at 2 a.m.")

Bomb #2 deserves its own demonstration, because it decides how every amount of money in
this project is written. Computers store decimals in binary, and most decimal fractions
don't fit exactly:

In [ ]:
print("0.1 + 0.2 =", 0.1 + 0.2, "   ← not 0.3")
print("Add a few billion micro-payments and the dust becomes real money.")
print("\nThe fix, industry-wide: money is an INTEGER count of the smallest unit.")
print("On our chain: 1 TOK = 10**18 base units, so 10 TOK =", 10 * 10**18)

A `dataclass` (chapter 03 used them for offers) fixes the *missing field* bomb — you
can't build one without all its fields — but it happily accepts nonsense **values**:
`capacity=-50` or `price="ten"` sail right through. What we want is a border guard that
checks *values*, not just names, the moment data arrives. That exists: **pydantic** — a
library where you declare each field's type and constraints, and construction either
returns a valid object or raises. Let's build a mini version of the project's deal
shape:

In [ ]:
from pydantic import BaseModel, Field, ValidationError, model_validator

class MiniOffer(BaseModel):
    model_config = {"frozen": True}          # like frozen dataclasses: no quiet edits
    provider: str
    capacity_bps: int = Field(gt=0)          # the unit is IN THE NAME, and it's positive
    price_base_units: int = Field(ge=0)      # integer money, no floats accepted
    start: int
    end: int

    @model_validator(mode="after")
    def window_must_run_forward(self):
        if self.end <= self.start:
            raise ValueError("end must be after start")
        return self

good = MiniOffer(provider="Bell", capacity_bps=50_000_000,
                 price_base_units=10 * 10**18,
                 start=1_757_944_800, end=1_757_952_000)
print("valid offer →", good)

In [ ]:
# Now feed it each bomb from the dict, and watch the border guard work:
attempts = [
    ("float price",      dict(provider="Bell", capacity_bps=50_000_000,
                              price_base_units=10.5, start=1, end=2)),
    ("negative capacity", dict(provider="Bell", capacity_bps=-50,
                              price_base_units=0, start=1, end=2)),
    ("window backwards",  dict(provider="Bell", capacity_bps=50_000_000,
                              price_base_units=0, start=2, end=1)),
]
for label, kwargs in attempts:
    try:
        MiniOffer(**kwargs)
    except ValidationError as e:
        print(f"{label:18} → rejected: {e.errors()[0]['msg']}")

Every bomb detonates **at the border, loudly**, instead of in production, quietly. Now
the reveal: the repo package `a2a_interfaces` is exactly this discipline applied to
*every* piece of data that crosses a package boundary — offers, entitlements, decisions,
resolved paths. One shared package of shapes, so Ada's code and Bell's code cannot
disagree about what an offer *is*. The canonical example you've been living with is
defined there, once, as validated objects:

In [ ]:
print("the canonical offer (chapter 03 settled exactly this):\n")
for field in ("provider", "consumer", "price", "start_time", "end_time", "valid_until"):
    print(f"  {field:14} = {getattr(fx.CANONICAL_OFFER, field)}")

print("\nfx.WINDOW        =", fx.WINDOW)
print("fx.PRICE_10_TOK  =", fx.PRICE_10_TOK, " ← integer money, as a digits-only string")

In [ ]:
# And the real Offer shape guards its border just like your MiniOffer did:
from a2a_interfaces.models import Offer

try:
    Offer(**{**fx.CANONICAL_OFFER.model_dump(), "provider": "Bell"})   # a name, not an address
except ValidationError as e:
    print("rejected:", e.errors()[0]["loc"], "—", e.errors()[0]["msg"])

`'Bell'` was good enough for our toys, but the real shapes insist on a real address —
`0x` + 40 hex characters — because that's what the chain can actually pay and verify.
From chapter 03 onward, whenever you see the toy use a name and the real system use an
address, this section is why.

> **🧭 Decision (pragmatic) — pydantic in one shared package, not a schema toolchain**
>
> Cross-package shapes could be defined in protobuf, JSON Schema, or an IDL with
> code generation — the standard answer when many languages and teams share a wire
> format. This prototype is one language (Python) end to end except the contract, so
> the simplest border guard that validates values — pydantic models in a single
> `a2a_interfaces` package every other package imports — demonstrates the discipline
> without the toolchain. Nothing in the architecture *depends* on pydantic; the rule
> that matters is "shapes are defined once, validated at the border."
> **In the paper:** implicitly §5 (implementation); worth one sentence at most.

**✏️ Your turn 3 — money with a decimal point**

Chapter 03's fine print said a decimal point never crosses the border. Prove the real
`Offer` enforces it: take `fx.CANONICAL_OFFER`'s fields and set `price` to `"10.5"`.
Predict the outcome, then run.

In [ ]:
# ...try to build an Offer whose price is "10.5"...

<details><summary>✅ Solution 3 — peek only after trying</summary>

```python
try:
    Offer(**{**fx.CANONICAL_OFFER.model_dump(), "price": "10.5"})
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])
```

Rejected — `price` is a *digits-only string* (`DecimalString` in
`a2a_interfaces.models`): integer base units, in a string so that enormous numbers
survive JSON round-trips unmangled. If you want half a TOK, you write
`"500000000000000000"` — the decimal point stays in the human's head, never in the
protocol.

</details>

## 4 · A first look at a real ledger (a cameo)

Everything in §1 you built with dicts and fingerprints; here's the genuine article for
sixty seconds — a real, disposable, private chain on your own machine. `anvil` (from
the Foundry toolkit) runs one; the repo's lab helper starts it and deploys the project's
contracts onto it. Chapter 03 puts this world to real work — today we just look:

In [ ]:
from chainmcp.testing import anvil_available, artifacts_available, launch_anvil

CHAIN_OK = anvil_available() and artifacts_available()
SKIP = ("skipped: needs anvil + built contracts — install Foundry "
        "(https://getfoundry.sh), then run:  forge build --root contracts")

anvil = None
print("anvil on PATH   →", "✓" if anvil_available() else "✗")
print("forge artifacts →", "✓" if artifacts_available() else "✗")
if not CHAIN_OK:
    print(SKIP)

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    from web3 import Web3

    anvil = launch_anvil(timestamp=fx.WINDOW.start - 900)     # born at 13:45, story time
    w3 = Web3(Web3.HTTPProvider(anvil.rpc_url))

    print("a real chain, just for you, at", anvil.rpc_url)
    print("chain id            →", w3.eth.chain_id)
    print("blocks so far       →", w3.eth.block_number, " (the deploys below)")
    print("Ada's gas balance   →", w3.eth.get_balance(fx.ADA) / 10**18, "ETH  (dev accounts start rich)")
    print("contracts deployed  →", anvil.deployment)
    print("MockTOK where fixtures promised?  →", anvil.deployment["MockTOK"] == fx.MOCK_TOK)

Note what just lined up: the *address of a contract* (`MockTOK`, the play-money token)
landed exactly where `fx.MOCK_TOK` said it would. Deployment on this chain is
deterministic — same deployer, same order, same addresses, every fresh world — which is
what lets one canonical example thread through story, docs, tests, and these notebooks.

In [ ]:
# Always clean up your disposable world — never orphan a chain process.
if anvil is not None:
    anvil.stop()
    print("world ended. Chapter 03 builds the next one and makes it earn its keep.")
else:
    print("nothing to clean up")

## 5 · What you can now say

- **What a ledger is** — a table of who has what — and why custody, not arithmetic, is
  its hard problem: whoever hosts it can edit it.
- **What hashing and chaining buy** — tamper-*evidence*, not tamper-*proofness*; the
  patient forger rewrites the tail, which is exactly why the chain must live on many
  machines that cross-check (and that's all the consensus depth this project uses).
- **What an account is** — a key pair, nothing more: secret key held close, derived
  address published, signatures checkable by anyone (mechanism: chapter 04).
- **Where keys live in this system** — in one custodian per agent; the controller
  verifies and never signs.
- **Why shapes are validated at the border** — units in field names, integer money,
  required fields, value constraints; `a2a_interfaces` is that discipline made into a
  package both agents must import.

## 6 · 📝 For the paper

This chapter is background and boundary material — it feeds **§2.3** (the smart-contract
primer a non-blockchain reader needs) and one structural sentence of **§4.6**. Draft
sentences you can defend:

| you can write… | because you ran… |
|---|---|
| *The ledger's integrity model is standard: a hash-chained history replicated across nodes; the prototype relies on it as a commodity and contributes nothing to consensus itself.* | §1's chain build, the patient-forger exercise, and its moral |
| *Private keys live in exactly one component per agent — its key custodian; every other component, including the controller, verifies signatures but never holds a key.* | §2's custody decision box; enforced in code, exercised from ch. 03's clients onward |
| *All cross-component data shapes are defined once, in a shared interface package, and validated at each border; malformed or unit-ambiguous terms are rejected before any component acts on them.* | §3's three bombs, caught by your MiniOffer and by the real `Offer` |

**Reviewer objections you can now answer:** *"why should anyone trust your ledger?"* —
we don't ask them to trust ours; we assume a standard replicated chain and study what's
*built on it* (and say so in §8.4). *"what if an agent is compromised?"* — it holds no
key; the blast radius is its custodian's refusal surface, not the wallet.

**Honesty inventory for §8.4:** single local Anvil node — instant mining, no consensus
latency, no fee market; public-chain figures in the paper are extrapolations. No
cross-language schema story — one-language prototype.

*Next: [03 — The atomic swap](03_the_atomic_swap.ipynb) — chapter 02 of the old plan
was merged into this one, so the numbering jumps.*